In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import xarray as xr
from glob import glob
import os
from netCDF4 import Dataset
import pandas as pd
from datetime import datetime, date, timedelta
from pathlib import Path
import scipy
import scipy.ndimage
from mpl_toolkits.axes_grid1 import ImageGrid
import math
import sys
import gc
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Configurations ---
input_dir = Path("/mnt/stor-pool-01/projects/heus/ShellAnalysis/full-area")
source_input_dir = Path("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/")

output_dir = Path("/mnt/stor-pool-01/projects/heus/ShellAnalysis/full-area")
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
input_file_name = "slab_averages_grouped.nc"
file_path = input_dir / input_file_name
if not file_path.is_file():
    print(f"❌ ERROR: Simulation dataset not found at:\n   {file_path}\nEnding program early.", file=sys.stderr)
    sys.exit(1)



In [ ]:
#loading ds_ql just for the shape
path_ds_ql = source_input_dir / "ql.nc"
if not path_ds_ql.is_file():
    print(f"❌ ERROR: Dataset missing: {path_ds_ql}", file=sys.stderr)
    sys.exit(1)

print("Interpolating w onto the core coordinate grid...")
ds_ql = xr.open_dataset(path_ds_ql, decode_times=False).load()

# 2. Extract spatial metadata bounds
num_times = int(ds_ql.time.size)
nz, ny, nx = ds_ql.ql.shape[1:]
z_coordinates = ds_ql.z.values
time_values = ds_ql.time.values

In [ ]:
#Iteration list setup

physical_vars = {
    "b": "b", "w": "w", "vpg": "vpg", "vpg_b": "vpg_b", "vpg_dn": "vpg_dn", "vpg_dl": "vpg_dl",
    "pi_b": "pi_b", "pi_dn": "pi_dn", "pi_dl": "pi_dl",
    "ke_b": "ke_b", "ke_b_eff": "ke_b_eff", "ke_vpg": "ke_vpg", "ke_vpg_b": "ke_vpg_b",
    "ke_vpg_dn": "ke_vpg_dn", "ke_vpg_dl": "ke_vpg_dl", "ke_w": "ke_w", "b_eff": "b_eff"
}

mask_keys = ["domain", "shell", "congestus", "deep", "congestus_shell", "deep_shell"]

# Initialize a nested tracking structure: { variable_group: { mask_name: DataArray } }
output_groups = {var_key: {} for var_key in physical_vars.keys()}

for var_key in physical_vars.keys():
    for m_key in mask_keys:
        output_groups[var_key][m_key] = xr.DataArray(
            data=np.full((nz), np.nan, dtype=np.float32),
            dims=("z"),
            coords={"z": z_coordinates},
            name=m_key
        )

In [ ]:
i = 1
arr_length = len(physical_vars)
for group_name, internal_name in physical_vars.items():
    print(f"Processing {group_name} ({i}/{arr_length})")
    plotted_group = xr.open_dataset(file_path, group=group_name, decode_times=False)

    for mask_type in mask_keys:
        print(f"    {group_name} ({i}/{arr_length}): Averaging on {mask_type}")
        output_groups[group_name][mask_type].values = plotted_group[mask_type].mean(dim='t')

    i += 1
    gc.collect()

In [ ]:
print(f"Exporting dataset...")
# Combine the complete dictionary of arrays into a centralized dataset
output_file = output_dir / "time_averaged_slab_averages_grouped.nc"

# To avoid conflicts, if an old run file exists, remove it before starting clean group writes
if output_file.exists():
    output_file.unlink()

# Loop through each physical variable and write it as its own isolated group
for var_key, mask_arrays in output_groups.items():
    print(f"Writing group: '{var_key}'...")
    
    # Convert this specific variable's masks into a tidy standalone dataset
    ds_group = xr.Dataset(mask_arrays)
    
    # Write to NetCDF under a specific group path
    ds_group.to_netcdf(output_file, mode="a", group=var_key)

print("\nAll computation and exporting complete (Program is safe to close)")